Visualize HSP2 Outputs
====

This Jupyter Notebook demonstrates interactive visualizations of HSP2 outputs.

Visualizations are created using [HoloViz](https://holoviz.org) suite of integrated Python packages designed to make viz easier, more accurate, and more powerful. 

HoloViz tools build on the SciPy/PyData/PyViz ecosystem.

![Familiar and high-level API for data exploration and visualization](https://hvplot.holoviz.org/assets/diagram.svg)


## Required Python Imports and Setup

In [1]:
import os
from pathlib import Path

import pandas as pd

In [2]:
import holoviews as hv
import hvplot.pandas
from holoviews import opts
from holoviews.plotting.links import RangeToolLink

hv.extension("bokeh")
hv.renderer("bokeh").webgl = False

In [3]:
# Confirm that your active environment for this notebook is the one you created for HSP2.
os.environ["CONDA_DEFAULT_ENV"]

'default311'

## Access Results

Note: you must run the model before executing this code. If you get an error, go to `NOTEBOOK NAME` to run the model. 

In [4]:
# Set your project directory to your local folder for your clone of the HSPsquared repository
project_folder = Path.cwd().parent
project_folder

PosixPath('/var/home/timcera/programming/HSP_old/HSPsquared')

In [5]:
# Set your temporary data output folder path
output_data_folder = Path("examples/_TutorialData")

In [6]:
# Create path for the temporary data output file you created with `1_Intro_to_HSP2.ipynb`

output_file = "test10.h5"  # HSP2 data HDF5 binary file, for all inputs and outputs

output_hdf5_path = (
    project_folder / output_data_folder / output_file
)  # pathlib concatenation

print(output_hdf5_path)
print("File exists? " + str(output_hdf5_path.exists()))
# The file path should not yet exist. If it does, delete it for this tutorial.

/var/home/timcera/programming/HSP_old/HSPsquared/examples/_TutorialData/test10.h5
File exists? True


We will access data from reach 5, which is the furthest downstream. Below, we access output on hydrology, sediment transport, and nutrients from this reach section. 

In [7]:
ts_hydrology = pd.read_hdf(output_hdf5_path, "/RESULTS/RCHRES_R005/HYDR")
ts_sedimentt = pd.read_hdf(output_hdf5_path, "/RESULTS/RCHRES_R005/SEDTRN")
ts_nutrients = pd.read_hdf(output_hdf5_path, "/RESULTS/RCHRES_R005/NUTRX")

In [8]:
ts_hydrology.head()

,AVDEP,AVVEL,DEP,HRAD,IVOL,POTEV,PRSUPY,RO,ROVOL,SAREA,TAU,TWID,USTAR,VOL,VOLEV
1976-01-01 01:00:00,0.702923,1.857768,0.822988,0.562775,1.747608,0.000583,0.0,7.371986,0.304628,2.052829,0.088680,5.645279,0.213918,1.442980,0.000000
1976-01-01 02:00:00,0.651220,1.770455,0.754730,0.526685,0.428585,0.000583,0.0,6.350832,0.567059,2.003019,0.082993,5.508303,0.206945,1.304407,0.000100
1976-01-01 03:00:00,0.534142,1.576605,0.604331,0.443217,0.150780,0.000583,0.0,4.385298,0.443642,1.893593,0.069840,5.207382,0.189840,1.011448,0.000097
1976-01-01 04:00:00,0.429286,1.416886,0.474719,0.365824,0.066827,0.000583,0.0,3.010308,0.305604,1.799683,0.057645,4.949128,0.172471,0.772579,0.000092
1976-01-01 05:00:00,0.346840,1.221629,0.376407,0.302669,0.034621,0.000583,0.0,2.013996,0.207616,1.728451,0.047693,4.753240,0.156879,0.599497,0.000087


## Visualize Data


### Easy plotting with `.hvplot()`

The [hvPlot](https://hvplot.holoviz.org/index.html) library provides easy interactive plotting for Pandas dataframes, via  `.hvplot()` methods that are more versatile and powerful than the standard `.plot()` API.

Documentation:
- https://hvplot.holoviz.org/user_guide/Plotting.html
- https://hvplot.holoviz.org/user_guide/Viewing.html

In [9]:
# Quick plots using default arguments
ts_hydrology.RO.hvplot()

:Curve   [index]   (RO)

In [10]:
# Plot an "Overlay" of many series at a time
ts_hydrology.hvplot(y=["RO", "ROVOL"])

:NdOverlay   [Variable]
   :Curve   [index]   (value)

In [11]:
# Plot a "Layout" of two plots, stacked in 1 column, using features from
# the more powerful Holoviews: https://holoviews.org/user_guide/Composing_Elements.html

(ts_hydrology.RO.hvplot() + ts_hydrology.ROVOL.hvplot()).cols(1)

:Layout
   .Curve.RO    :Curve   [index]   (RO)
   .Curve.ROVOL :Curve   [index]   (ROVOL)

### Hydrology

The [HoloViews](https://holoviews.org/index.html) library let's you access the under-the-hood power over how your data gets plotted, and serves as the core of the HoloViz plotting system.

Examples below provide a gallery of what is possible with only a few lines of code.

Documentation:
- https://holoviews.org/getting_started/Introduction.html

In [12]:
ts_hydrology.index.name = "Date"
flow_curve = hv.Curve(ts_hydrology, "Date", ("RO", "Outflow (cms)"))

In [13]:
# set up timeseries
tgt = flow_curve.relabel("Reach 5 Outflow").opts(
    width=600,
    height=300,
    tools=["wheel_zoom", "box_zoom", "hover", "pan", "reset"],
    toolbar="above",
)
# set up zoom bar
src = flow_curve.opts(width=600, height=100, yaxis=None, default_tools=[])

# link timeseries and zoom bar
RangeToolLink(src, tgt)

layout = (tgt + src).cols(1)
layout.opts(opts.Layout(shared_axes=False, merge_tools=False))

:Layout
   .Curve.Reach_5_Outflow :Curve   [Date]   (Outflow (cms))
   .Curve.I               :Curve   [Date]   (Outflow (cms))

The following function can allow you to plot any hydrological output for any of the reaches in the model. 
* `rch_no`: reach number to plot (**int**)
* `constituent`: variable to plot, default = 'RO' (**str**)

In [14]:
# function
def PlotHydrology(rch_no, constituent="RO"):
    ts_hydrology = pd.read_hdf(output_hdf5_path, f"/RESULTS/RCHRES_R{rch_no:03}/HYDR")
    ts_hydrology.index.name = "Date"
    flow_curve = hv.Curve(ts_hydrology, "Date", constituent)
    # set up timeseries
    tgt = flow_curve.relabel("Reach %d" % rch_no).opts(
        width=600,
        height=300,
        #   tools = ['wheel_zoom', 'box_zoom', 'hover', 'pan', 'reset'],
        toolbar="above",
    )
    # set up zoom bar
    src = flow_curve.opts(width=600, height=100, yaxis=None, default_tools=[])

    # link timeseries and zoom bar
    RangeToolLink(src, tgt)

    layout = (tgt + src).cols(1)
    return layout.opts(opts.Layout(shared_axes=False, merge_tools=False))

Try the function to plot outflows from Reach 4:

In [15]:
PlotHydrology(1)

:Layout
   .Curve.Reach_1 :Curve   [Date]   (RO)
   .Curve.I       :Curve   [Date]   (RO)

Use the function to plot the Volume of water contributed by precipitation:

In [16]:
PlotHydrology(4, constituent="PRSUPY")

:Layout
   .Curve.Reach_4 :Curve   [Date]   (PRSUPY)
   .Curve.I       :Curve   [Date]   (PRSUPY)

### Nutrients

* Total Inflows: `TNUIF1_n` 
* Total Outflows: `TNUIF_n`

where `n` is a number between 1 and 5 specifying the nutrient of interest:
* 1=NO3
* 2=TAM
* 3=NO2
* 4=PO4
* 5=NH4+
* 6=NH3

In [17]:
ts_nutrients.index.name = "Date"
nutrient_inflow_curve = hv.Curve(
    ts_nutrients, "Date", ("TNUIF1", "Total NO3 (kg/ivld)"), label="inflow"
)
nutrient_outflow_curve = hv.Curve(
    ts_nutrients, "Date", ("TNUCF1_1", "Total NO3 (kg/ivld)"), label="outflow"
)

In [18]:
# hv.Overlay(list_of_curves).opts()
list_of_curves = [nutrient_inflow_curve, nutrient_outflow_curve]

# set up timeseries
source_curves, target_curves = [], []
for curve in list_of_curves:
    # Without relabel, the curve somehow shares the ranging properties. opts with clone=True doesn't help either.
    src = curve.relabel("").opts(width=600, height=100, yaxis=None, default_tools=[])
    tgt = curve
    source_curves.append(src)
    target_curves.append(tgt)

# link RangeTool for the first curves in the list.
RangeToolLink(source_curves[0], target_curves[0])

# Overlay the source and target curves
overlaid_plot_src = hv.Overlay(source_curves)
overlaid_plot_tgt = (
    hv.Overlay(list_of_curves)
    .relabel("Reach 5 NO3")
    .opts(
        width=600,
        height=300,
        tools=["wheel_zoom", "box_zoom", "hover", "pan", "reset"],
        toolbar="above",
        ylim=(-0.5, 11),
    )
)

# layout the plot and render
layout = (overlaid_plot_tgt + overlaid_plot_src).cols(1)
layout.opts(merge_tools=False, shared_axes=False)

:Layout
   .Overlay.Reach_5_NO3 :Overlay
      .Curve.Inflow  :Curve   [Date]   (Total NO3 (kg/ivld))
      .Curve.Outflow :Curve   [Date]   (Total NO3 (kg/ivld))
   .Overlay.I           :Overlay
      .Curve.I  :Curve   [Date]   (Total NO3 (kg/ivld))
      .Curve.II :Curve   [Date]   (Total NO3 (kg/ivld))

### Sediment Transport

* `ISED_n`: sum of inflows of sediment
* `ROSEDn`: sum of outflows of sediment 

where `n` defines the sediment of interest:
* 1 for sand
* 2 for silt
* 3 for clay
* 4 for the sum of sand silt and clay
* The subscript with maximum value =10 selects the following:  1 suspended sand, 2 suspended silt, 3 suspended clay, 4 bed sand, 5 bed silt 6 bed clay, 7 total sand, 8 total silt, 9 total clay, and 10 total of 7,8,9.


In [19]:
ts_sedimentt.index.name = "Date"
sediment_inflow_curve = hv.Curve(
    ts_sedimentt, "Date", ("ISED1", "Total Sand (ton/ivld)"), label="inflow"
)
sediment_outflow_curve = hv.Curve(
    ts_sedimentt, "Date", ("ROSED1", "Total Sand (ton/ivld)"), label="outflow"
)

In [20]:
# hv.Overlay(list_of_curves).opts()
list_of_curves = [sediment_inflow_curve, sediment_outflow_curve]

# set up timeseries
source_curves, target_curves = [], []
for curve in list_of_curves:
    # Without relabel, the curve somehow shares the ranging properties. opts with clone=True doesn't help either.
    src = curve.relabel("").opts(width=600, height=100, yaxis=None, default_tools=[])
    tgt = curve
    source_curves.append(src)
    target_curves.append(tgt)

# link RangeTool for the first curves in the list.
RangeToolLink(source_curves[0], target_curves[0])

# Overlay the source and target curves
overlaid_plot_src = hv.Overlay(source_curves)
overlaid_plot_tgt = (
    hv.Overlay(list_of_curves)
    .relabel("Reach 5 Sand")
    .opts(
        width=600,
        height=300,
        tools=["wheel_zoom", "box_zoom", "hover", "pan", "reset"],
        toolbar="above",
        ylim=(-0.5, 7),
    )
)

# layout the plot and render
layout = (overlaid_plot_tgt + overlaid_plot_src).cols(1)
layout.opts(merge_tools=False, shared_axes=False)

:Layout
   .Overlay.Reach_5_Sand :Overlay
      .Curve.Inflow  :Curve   [Date]   (Total Sand (ton/ivld))
      .Curve.Outflow :Curve   [Date]   (Total Sand (ton/ivld))
   .Overlay.I            :Overlay
      .Curve.I  :Curve   [Date]   (Total Sand (ton/ivld))
      .Curve.II :Curve   [Date]   (Total Sand (ton/ivld))